# RAG Retrieval Accuracy — Sliced Dual Embedding Comparison

Compares three query strategies on the val split:

| Strategy | Query vector | Training index |
|---|---|---|
| **full_dual** | `normalise(α·embed(raw) + (1-α)·embed(30_facets))` | `essays_dual/` |
| **sliced_dual_asym** | `normalise(α·embed(raw) + (1-α)·embed(6_facets))` | `essays_dual/` (full-dual) |
| **sliced_dual_sym** | `normalise(α·embed(raw) + (1-α)·embed(6_facets))` | `essays_sliced_dual/{trait}/` |

Strategy C (symmetric) requires running `build_sliced_dual_index.ipynb` first.
If the per-trait indexes are missing, only A and B are evaluated.

**Prerequisites:**
- `data/vector_db/essays_dual/` — run `build_hybrid_index.ipynb`
- `data/profile_db/essays_val/` — run `rag_retrieve_accuracy_profile.ipynb`
- `data/vector_db/essays_sliced_dual/` *(optional)* — run `build_sliced_dual_index.ipynb`

In [1]:
import os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
VAL_CSV            = str(project_root / "data/split/essays/val.csv")
VAL_PROFILE_DB     = str(project_root / "data/profile_db/essays_val")
DUAL_DB            = str(project_root / "data/vector_db/essays_dual")
SLICED_DUAL_BASE   = str(project_root / "data/vector_db/essays_sliced_dual")
RES_DIR            = str(project_root / "result/retrieve_accuracy/sliced_dual")

K_VALUES = [3, 5]

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

val_df = pd.read_csv(VAL_CSV)
print(f"Val split          : {len(val_df)} rows")

for label, p in [("Dual DB", DUAL_DB), ("Val profiles", VAL_PROFILE_DB)]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:<18s}: [{status}]  {p}")

sym_available = Path(SLICED_DUAL_BASE).exists() and all(
    (Path(SLICED_DUAL_BASE) / tc / "vectors.faiss").exists() for tc in TRAITS
)
print(f"  {'Sliced dual DBs':<18s}: {'OK — strategy C enabled' if sym_available else 'MISSING — strategy C will be skipped'}")

if not Path(DUAL_DB).exists():
    raise RuntimeError(f"Missing dual index. Run notebook/gpt/build_hybrid_index.ipynb first.")

Val split          : 247 rows
  Dual DB           : [OK]  F:\std\GR\code\model_x_ocean\data\vector_db\essays_dual
  Val profiles      : [OK]  F:\std\GR\code\model_x_ocean\data\profile_db\essays_val
  Sliced dual DBs   : MISSING — strategy C will be skipped


## Step 1 — Load val profiles

In [3]:
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS

val_store_path = Path(VAL_PROFILE_DB) / "profile_store.jsonl"
if not val_store_path.exists():
    raise RuntimeError(
        f"Val profiles not found at {val_store_path}.\n"
        "Run notebook/rag_compare/rag_retrieve_accuracy_profile.ipynb to build them."
    )

val_store = ProfileStore(str(val_store_path))
val_store.load()
val_profiles = {
    int(e["user_id"].split("_")[1]): e
    for e in val_store.get_all() if e.get("valid")
}
print(f"Val profiles: {len(val_profiles)} / {len(val_df)}")

Val profiles: 247 / 247


## Step 2 — Load FAISS indexes

In [4]:
from rag.faiss_index import FAISSIndex

# Full dual index (strategies A and B)
dual_index = FAISSIndex(dimension=0)
dual_index.load(
    str(Path(DUAL_DB) / "vectors.faiss"),
    str(Path(DUAL_DB) / "vectors_meta.jsonl"),
)
print(f"Dual index: {dual_index._index.ntotal} vectors, dim={dual_index.dimension}")

# Per-trait sliced dual indexes (strategy C — optional)
sliced_indexes = {}
if sym_available:
    for trait_code in TRAITS:
        idx = FAISSIndex(dimension=0)
        idx.load(
            str(Path(SLICED_DUAL_BASE) / trait_code / "vectors.faiss"),
            str(Path(SLICED_DUAL_BASE) / trait_code / "vectors_meta.jsonl"),
        )
        sliced_indexes[trait_code] = idx
    print(f"Per-trait indexes loaded: {list(sliced_indexes.keys())}")
else:
    print("Per-trait indexes not found — strategy C skipped.")
    print("Run notebook/gpt/build_sliced_dual_index.ipynb to enable it.")

Dual index: 1974 vectors, dim=768
Per-trait indexes not found — strategy C skipped.
Run notebook/gpt/build_sliced_dual_index.ipynb to enable it.


## Step 3 — Compute val query embeddings

Encoding schedule (minimises redundant calls):
- Raw posts: **1×** (shared by all strategies)
- Full profiles: **1×** (strategy A)
- Sliced profiles: **5×** one per trait (strategies B and C)

In [5]:
from rag.embedder         import _embed_single, get_embedding, ALPHA
from rag.profiler.prompts import slice_profile_for_trait


def render_full_profile_text(entry: dict) -> str:
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines  = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


def _fuse(raw_emb: np.ndarray, prof_emb: np.ndarray, alpha: float = ALPHA) -> np.ndarray:
    fused = alpha * raw_emb + (1.0 - alpha) * prof_emb
    norm  = np.linalg.norm(fused)
    return (fused / norm) if norm > 0 else fused


N = len(val_df)

# --- Raw embeddings ---
print(f"Encoding {N} raw posts ...")
t0 = time.time()
raw_embs = np.array(
    [_embed_single(str(row["text"])) for _, row in val_df.iterrows()],
    dtype="float32",
)
print(f"Done in {time.time()-t0:.1f}s  shape={raw_embs.shape}")

# --- Full profile embeddings (strategy A) ---
full_profile_texts = [
    render_full_profile_text(val_profiles[i]) if i in val_profiles else str(val_df.iloc[i]["text"])
    for i in range(N)
]
print(f"\nEncoding {N} full profiles ...")
t0 = time.time()
full_prof_embs = np.array(get_embedding(full_profile_texts), dtype="float32")
print(f"Done in {time.time()-t0:.1f}s")

full_dual_embs = np.array(
    [_fuse(raw_embs[i], full_prof_embs[i]) for i in range(N)],
    dtype="float32",
)

# --- Sliced profile embeddings per trait (strategies B and C) ---
sliced_dual_embs = {}   # trait_code -> np.ndarray (N, 768)

for trait_code, trait_full in TRAITS.items():
    sliced_texts = []
    for i, (_, row) in enumerate(val_df.iterrows()):
        pe = val_profiles.get(i)
        t  = slice_profile_for_trait(pe, trait_code) if pe else ""
        sliced_texts.append(t if t.strip() else str(row["text"]))

    print(f"\nEncoding {N} sliced profiles [{trait_code}] ...")
    t0 = time.time()
    sp_embs = np.array(get_embedding(sliced_texts), dtype="float32")
    print(f"Done in {time.time()-t0:.1f}s")

    sliced_dual_embs[trait_code] = np.array(
        [_fuse(raw_embs[i], sp_embs[i]) for i in range(N)],
        dtype="float32",
    )

print("\nAll embeddings ready.")

Encoding 247 raw posts ...
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


Done in 348.1s  shape=(247, 768)

Encoding 247 full profiles ...
Done in 275.1s

Encoding 247 sliced profiles [cOPN] ...
Done in 61.4s

Encoding 247 sliced profiles [cCON] ...
Done in 64.0s

Encoding 247 sliced profiles [cEXT] ...
Done in 52.1s

Encoding 247 sliced profiles [cAGR] ...
Done in 45.0s

Encoding 247 sliced profiles [cNEU] ...
Done in 49.9s

All embeddings ready.


## Step 4 — Evaluate retrieval accuracy

In [6]:
def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s


def run_eval(strategy_name: str, get_query_emb, get_index=None):
    """
    get_query_emb(i, trait_code) -> np.ndarray (768,)
    get_index(trait_code)        -> FAISSIndex  [default: dual_index]
    """
    _get_idx = get_index if get_index is not None else (lambda tc: dual_index)
    rows = []
    for k in K_VALUES:
        print(f"\n{'='*60}")
        print(f"  strategy={strategy_name!r}  k={k}")
        print(f"{'='*60}")
        for trait_code, trait_full in TRAITS.items():
            index = _get_idx(trait_code)
            match_rates, hits = [], []
            for i, (_, row) in enumerate(val_df.iterrows()):
                true_label = normalize_label(row[trait_code])
                if true_label not in ("high", "low"):
                    continue

                q_emb      = get_query_emb(i, trait_code)
                _, results = index.search(q_emb, k * 6)
                filtered   = [
                    r for r in results
                    if trait_full in r.get("trait_labels", {})
                ][:k]

                n       = len(filtered)
                matches = sum(1 for r in filtered if r["trait_labels"][trait_full] == true_label)
                match_rates.append(matches / n if n > 0 else 0.0)
                hits.append(int(matches > 0))

            mean_r = float(np.mean(match_rates))
            std_r  = float(np.std(match_rates))
            hit_r  = float(np.mean(hits))
            print(f"  {trait_full:<30s}  match_rate={mean_r:.4f} ± {std_r:.4f}  hit_rate={hit_r:.4f}")
            rows.append({
                "strategy":        strategy_name,
                "trait":           trait_full,
                "k":               k,
                "n_queries":       len(match_rates),
                "mean_match_rate": mean_r,
                "std_match_rate":  std_r,
                "hit_rate":        hit_r,
            })
    return rows


all_rows = []

# Strategy A — full dual query, full dual index
all_rows += run_eval("full_dual",        lambda i, tc: full_dual_embs[i])

# Strategy B — sliced dual query, full dual index (asymmetric)
all_rows += run_eval("sliced_dual_asym", lambda i, tc: sliced_dual_embs[tc][i])

# Strategy C — sliced dual query, per-trait sliced index (symmetric)
if sliced_indexes:
    all_rows += run_eval(
        "sliced_dual_sym",
        lambda i, tc: sliced_dual_embs[tc][i],
        get_index=lambda tc: sliced_indexes[tc],
    )
else:
    print("\n[Strategy C skipped — per-trait indexes not found]")

summary_df = pd.DataFrame(all_rows)
print("\nEvaluation complete.")


  strategy='full_dual'  k=3
  Openness to Experience          match_rate=0.5169 ± 0.2886  hit_rate=0.8907
  Conscientiousness               match_rate=0.5155 ± 0.2902  hit_rate=0.8826
  Extraversion                    match_rate=0.5209 ± 0.2804  hit_rate=0.8988
  Agreeableness                   match_rate=0.5358 ± 0.2821  hit_rate=0.9190
  Neuroticism                     match_rate=0.5047 ± 0.3144  hit_rate=0.8462

  strategy='full_dual'  k=5
  Openness to Experience          match_rate=0.5174 ± 0.2441  hit_rate=0.9636
  Conscientiousness               match_rate=0.5304 ± 0.2270  hit_rate=0.9717
  Extraversion                    match_rate=0.5142 ± 0.2285  hit_rate=0.9636
  Agreeableness                   match_rate=0.5401 ± 0.2166  hit_rate=0.9879
  Neuroticism                     match_rate=0.5117 ± 0.2554  hit_rate=0.9433

  strategy='sliced_dual_asym'  k=3
  Openness to Experience          match_rate=0.5061 ± 0.3172  hit_rate=0.8462
  Conscientiousness               match_rate=0.5

## Step 5 — Display & compare

In [7]:
strategies = summary_df["strategy"].unique().tolist()

for k_val in K_VALUES:
    sub = summary_df[summary_df["k"] == k_val]

    pivot = sub.pivot_table(index="trait", columns="strategy", values="mean_match_rate")
    # delta columns vs full_dual baseline
    for s in strategies:
        if s != "full_dual" and "full_dual" in pivot.columns:
            pivot[f"Δ_{s}"] = pivot[s] - pivot["full_dual"]
    print(f"\n=== Mean match rate  (k={k_val}) ===")
    display(pivot.round(4))

    pivot_hit = sub.pivot_table(index="trait", columns="strategy", values="hit_rate")
    for s in strategies:
        if s != "full_dual" and "full_dual" in pivot_hit.columns:
            pivot_hit[f"Δ_{s}"] = pivot_hit[s] - pivot_hit["full_dual"]
    print(f"\n=== Hit rate  (k={k_val}) ===")
    display(pivot_hit.round(4))


=== Mean match rate  (k=3) ===


strategy,full_dual,sliced_dual_asym,Δ_sliced_dual_asym
trait,,,
Agreeableness,0.5358,0.5290,-0.0067
Conscientiousness,0.5155,0.5115,-0.0040
Extraversion,0.5209,0.5277,0.0067
Neuroticism,0.5047,0.5385,0.0337
Openness to Experience,0.5169,0.5061,-0.0108



=== Hit rate  (k=3) ===


strategy,full_dual,sliced_dual_asym,Δ_sliced_dual_asym
trait,,,
Agreeableness,0.9190,0.8947,-0.0243
Conscientiousness,0.8826,0.8340,-0.0486
Extraversion,0.8988,0.8704,-0.0283
Neuroticism,0.8462,0.8664,0.0202
Openness to Experience,0.8907,0.8462,-0.0445



=== Mean match rate  (k=5) ===


strategy,full_dual,sliced_dual_asym,Δ_sliced_dual_asym
trait,,,
Agreeableness,0.5401,0.5385,-0.0016
Conscientiousness,0.5304,0.5255,-0.0049
Extraversion,0.5142,0.5174,0.0032
Neuroticism,0.5117,0.5368,0.0251
Openness to Experience,0.5174,0.5198,0.0024



=== Hit rate  (k=5) ===


strategy,full_dual,sliced_dual_asym,Δ_sliced_dual_asym
trait,,,
Agreeableness,0.9879,0.9960,0.0081
Conscientiousness,0.9717,0.9595,-0.0121
Extraversion,0.9636,0.9514,-0.0121
Neuroticism,0.9433,0.9433,0.0000
Openness to Experience,0.9636,0.9595,-0.0040


In [8]:
os.makedirs(RES_DIR, exist_ok=True)
summary_path = os.path.join(RES_DIR, "accuracy_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Saved -> {summary_path}  ({len(summary_df)} rows)")

Saved -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\sliced_dual\accuracy_summary.csv  (20 rows)
